<a href="https://colab.research.google.com/github/1816x/Algoritmos-Aprendizaje-Automatico/blob/main/Practica%20Tema%2014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica Tema 14 Parte 1

**Clase:** Fundamentos Algoritmos de Aprendizaje Automatico

**Tema:** Documentacion Efectiva y Gobernanza de Modelos

## reto 1. entrenamiento reproducible con semilla fija y artefacto

aquí hacemos que el experimento sea reproducible. Fijamos la semilla en 42, entrenamos un Random Forest y guardamos la métrica F1 en un archivo JSON. La idea es que otra persona pueda correr lo mismo y obtener prácticamente el mismo resultado.

In [1]:
from sklearn.ensemble import RandomForestClassifier as random_forest_classifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.datasets import make_classification
import json
import numpy as np
from pathlib import Path

# fix the seed so the experiment can be repeated
np.random.seed(42)

# create a simulated classification dataset
x, y = make_classification(
    n_samples=500,
    n_features=6,
    random_state=42
)

# split the dataset using a fixed random state
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

# create the model with a fixed number of trees and seed
model = random_forest_classifier(
    n_estimators=50,
    random_state=42
)

# train the model
model.fit(x_train, y_train)

# calculate the f1 score on test data
f1_val = f1_score(y_test, model.predict(x_test))

# create the reproducibility report
report = {
    "seed": 42,
    "n_estimators": 50,
    "test_f1": float(f1_val)
}

# create the artifacts folder
Path("artefactos").mkdir(exist_ok=True)

# save the metrics as json
Path("artefactos/metrics.json").write_text(
    json.dumps(report, indent=2),
    encoding="utf-8"
)

print("base metrics exported:", report)


base metrics exported: {'seed': 42, 'n_estimators': 50, 'test_f1': 0.9}


## reto 2. auditoria de condiciones de reproducibilidad

aquí comprobamos si una segunda ejecución se parece lo suficiente a la original. Comparamos dos valores de F1 y aceptamos la réplica si la diferencia es menor a 0.02. Como 0.850 - 0.841 = 0.009, sí pasa la tolerancia.

In [2]:
# define the reference f1 and the second execution result
test_f1_a = 0.850
test_f1_b = 0.841

# define the accepted tolerance
epsilon = 0.02

# calculate the absolute difference
f1_difference = abs(test_f1_a - test_f1_b)

# approve or reject the replica
if f1_difference < epsilon:
    replica_status = "approved"
else:
    replica_status = "rejected"

print("f1 difference:", round(f1_difference, 4))
print("tolerance:", epsilon)
print("replica status:", replica_status)


f1 difference: 0.009
tolerance: 0.02
replica status: approved


La diferencia es 0.009, menor que 0.02. Por lo tanto, la segunda ejecucion se considera una replica aceptable.

## reto 3. creacion de bitacora de decisiones

aquí documentamos decisiones técnicas. Por ejemplo, por qué usamos 50 árboles y por qué fijamos la semilla 42. Esto sirve para que después no se pierda el contexto de por qué se configuró el modelo de cierta manera.

In [3]:
decision_log = """# decision log

| decision | justification | technical evidence |
|---|---|---|
| use n_estimators=50 | 50 trees provide enough model capacity without unnecessary computation | stable f1 score with lower training cost |
| use random_state=42 | a fixed seed improves reproducibility | repeated executions keep the same random behavior |
"""

Path("artefactos/DECISION_LOG.md").write_text(
    decision_log,
    encoding="utf-8"
)

print(decision_log)


# decision log

| decision | justification | technical evidence |
|---|---|---|
| use n_estimators=50 | 50 trees provide enough model capacity without unnecessary computation | stable f1 score with lower training cost |
| use random_state=42 | a fixed seed improves reproducibility | repeated executions keep the same random behavior |



La bitacora permite entender por que se tomaron ciertas decisiones y facilita revisar o comparar futuras versiones del modelo.

## reto 4. matriz raci para promocion del modelo

aquí definimos quién hace qué cuando el modelo se va a promover a producción. el especialista técnico ejecuta, el propietario del modelo aprueba y cumplimiento/ética revisa riesgos. La idea es que quien ejecuta el código no sea la misma persona que lo aprueba, para tener una revisión independiente.

In [4]:
# define the raci responsibilities for the promotion process
raci_matrix = {
    "model_owner": "A",
    "technical_specialist": "R",
    "compliance_ethics": "C"
}

for role, responsibility in raci_matrix.items():
    print(role, "->", responsibility)

print("other stakeholders -> I")


model_owner -> A
technical_specialist -> R
compliance_ethics -> C
other stakeholders -> I


**Asignacion propuesta:**

- **Propietario del modelo — A:** aprueba finalmente la promocion.
- **Especialista tecnico — R:** ejecuta validaciones y despliegue.
- **Cumplimiento/etica — C:** revisa riesgos y politicas.
- **Otras partes interesadas — I:** reciben informacion sobre la decision o despliegue.

El aprobador no deberia ser la misma persona que ejecuta el codigo porque una revision independiente reduce errores y conflictos de interes.

## conclusion

La practica muestra como asegurar trazabilidad y gobernanza: reproducir resultados, documentar decisiones tecnicas y separar claramente responsabilidades antes de promover un modelo a produccion.